# Context Strategy Evaluation

Compares five context strategies on LongBench using Gemini 2.5 Flash as the answer model.

**Strategies evaluated:**
- `full_context` — baseline, passes full document to answer model
- `retrieval` — TF-IDF top-k chunk selection
- `compression` — LLMLingua-2 token-level compression
- `gemini_summarization` — query-aware summary via Gemini Flash
- `retrieval_compression` — retrieve top-k chunks, then compress

**To split work across multiple people:** each person sets a different `OFFSET` below. Results can be combined at the end.

## 1. Authenticate

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from huggingface_hub import login
login()

## 2. Clone Repo

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/rissingh23/Context-Opt.git'
REPO_DIR = Path('/content/Context-Opt')
BRANCH = 'gemini-eval'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git checkout {BRANCH}
print('Working directory:', Path.cwd())

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Config

**Set these before running:**
- `GCP_PROJECT` — your GCP project ID
- `OFFSET` — which examples to run. Use different offsets to split work:
  - Person 1: `OFFSET=0, LIMIT=50`
  - Person 2: `OFFSET=50, LIMIT=50`
  - Person 3: `OFFSET=100, LIMIT=50`
- `RUN_NAME` — a unique name for your output files (e.g. your name or `run1`)

In [ ]:
GCP_PROJECT  = 'contextops-498110'
GCP_LOCATION = 'us-central1'
GEMINI_MODEL = 'gemini-2.5-flash'

# --- Split config: each person uses a different OFFSET ---
# Person 1: OFFSET=0,   LIMIT=50, RUN_NAME='run_rajit'
# Person 2: OFFSET=50,  LIMIT=50, RUN_NAME='run_friend1'
# Person 3: OFFSET=100, LIMIT=50, RUN_NAME='run_friend2'
OFFSET   = 0
LIMIT    = 50
RUN_NAME = 'run1'

# --- Utility function weights ---
# Utility = Quality - λ·(Token Cost) - β·(Latency)
# Increase λ to penalise expensive strategies more
# Increase β to penalise slow strategies more
LAMBDA_COST  = 100.0   # penalty per dollar of token cost
BETA_LATENCY = 0.001   # penalty per second of latency

# --- Judge: set to False to skip LLM scoring (faster, no extra API calls) ---
USE_JUDGE = True

TASKS      = 'qasper hotpotqa gov_report multi_news passage_count'
STRATEGIES = 'full_context retrieval compression gemini_summarization retrieval_compression'

ROWS_OUT = f'outputs/processed/{RUN_NAME}_rows.csv'
AGG_OUT  = f'outputs/processed/{RUN_NAME}_summary.csv'
CKPT_OUT = f'outputs/processed/{RUN_NAME}_checkpoint.jsonl'

JUDGE_FLAGS = f'--judge vertexai --judge-model {GEMINI_MODEL} --quality-source llm_judge' if USE_JUDGE else '--judge disabled --quality-source automatic'

print(f'Running examples {OFFSET} to {OFFSET + LIMIT} per task')
print(f'Utility = Quality - {LAMBDA_COST}·Cost - {BETA_LATENCY}·Latency')
print(f'Judge: {"Gemini" if USE_JUDGE else "disabled (automatic metrics only)"}')
print(f'Outputs: {ROWS_OUT}')

In [ ]:
# Verify Gemini connection
from google import genai
client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)
response = client.models.generate_content(model=GEMINI_MODEL, contents='Reply with OK.')
print('Gemini connection:', response.text)

## 5. Download LongBench Data

Downloads all 200 test examples per task. Only needs to run once per session.

In [ ]:
!python3 -m src.data.download_longbench \
  --tasks {TASKS}

## 6. Smoke Test — 1 Example

Run this first to confirm all strategies work before the full eval.

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper \
  --strategies {STRATEGIES} \
  --limit 1 \
  --offset 0 \
  --provider vertexai \
  --model {GEMINI_MODEL} \
  {JUDGE_FLAGS} \
  --vertexai-project {GCP_PROJECT} \
  --vertexai-location {GCP_LOCATION} \
  --rows-output outputs/processed/smoke_rows.csv \
  --aggregate-output outputs/processed/smoke_summary.csv \
  --json-output outputs/processed/smoke_checkpoint.jsonl

In [ ]:
import pandas as pd
pd.read_csv('outputs/processed/smoke_summary.csv')[[
    'task', 'strategy', 'num_examples', 'error_rate',
    'avg_quality_score', 'avg_compression_ratio', 'avg_total_latency_sec'
]]

## 7. Full Eval

Runs your assigned slice of examples across all tasks and strategies.

**Checkpoint/resume:** results are saved after every single example to `CKPT_OUT`. If the run is interrupted, just rerun this cell — it will skip already-completed rows and pick up where it left off.

**Expected warnings (all harmless):**
- `Token indices sequence length is longer than...` — LLMLingua truncates long docs internally
- `FutureWarning: _check_is_size` — bitsandbytes internal warning
- `The following generation flags are not valid` — Gemini/Qwen config warning

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks {TASKS} \
  --strategies {STRATEGIES} \
  --limit {LIMIT} \
  --offset {OFFSET} \
  --provider vertexai \
  --model {GEMINI_MODEL} \
  {JUDGE_FLAGS} \
  --vertexai-project {GCP_PROJECT} \
  --vertexai-location {GCP_LOCATION} \
  --lambda-cost {LAMBDA_COST} \
  --beta-latency {BETA_LATENCY} \
  --rows-output {ROWS_OUT} \
  --aggregate-output {AGG_OUT} \
  --json-output {CKPT_OUT}

## 8. View Results

In [ ]:
import pandas as pd\n",
    "\n",
    "df = pd.read_csv(AGG_OUT)\n",
    "df[[\n",
    "    'task', 'task_type', 'strategy',\n",
    "    'num_examples', 'error_rate',\n",
    "    'avg_llm_judge_score', 'avg_quality_score',\n",
    "    'avg_rouge_l', 'avg_token_f1',\n",
    "    'avg_compression_ratio',\n",
    "    'avg_total_latency_sec',\n",
    "    'avg_estimated_cost',\n",
    "    'avg_utility_score',\n",
    "]].sort_values(['task', 'avg_utility_score'], ascending=[True, False])

In [ ]:
# Check for errors
rows = pd.read_csv(ROWS_OUT)
errors = rows[rows['error'].fillna('') != '']
print(f'Error rows: {len(errors)} / {len(rows)}')
if len(errors):
    print(errors[['task', 'strategy', 'error']].to_string())

## 9. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = Path(f'/content/drive/MyDrive/context-opt-results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

!cp {ROWS_OUT} {DRIVE_OUT}/{RUN_NAME}_rows.csv
!cp {AGG_OUT} {DRIVE_OUT}/{RUN_NAME}_summary.csv
!cp {CKPT_OUT} {DRIVE_OUT}/{RUN_NAME}_checkpoint.jsonl

print(f'Saved to {DRIVE_OUT}')

## 10. Combine Results from Multiple Runs

Once all team members have saved their results to Drive, run this to merge and aggregate everything.

In [ ]:
import pandas as pd
import json
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/context-opt-results')

# Load all checkpoint files from Drive
all_rows = []
for ckpt in sorted(DRIVE_OUT.glob('*_checkpoint.jsonl')):
    with ckpt.open() as f:
        for line in f:
            if line.strip():
                all_rows.append(json.loads(line))
    print(f'Loaded {ckpt.name}: {len(all_rows)} total rows so far')

# Deduplicate by (task, example_id, strategy)
seen = set()
deduped = []
for row in all_rows:
    key = (row['task'], row['example_id'], row['strategy'])
    if key not in seen:
        seen.add(key)
        deduped.append(row)

print(f'\nTotal unique rows: {len(deduped)}')
combined_df = pd.DataFrame(deduped)

# Aggregate
agg = combined_df.groupby(['task', 'strategy', 'model']).agg(
    num_examples=('example_id', 'count'),
    error_rate=('error', lambda x: (x.fillna('') != '').mean()),
    avg_quality_score=('quality_score', 'mean'),
    avg_rouge_l=('rouge_l', 'mean'),
    avg_token_f1=('token_f1', 'mean'),
    avg_compression_ratio=('compression_ratio', 'mean'),
    avg_total_latency_sec=('total_latency_sec', 'mean'),
    avg_estimated_cost=('estimated_cost', 'mean'),
).reset_index()

# Save combined results
combined_df.to_csv(DRIVE_OUT / 'combined_rows.csv', index=False)
agg.to_csv(DRIVE_OUT / 'combined_summary.csv', index=False)
print('Saved combined_rows.csv and combined_summary.csv to Drive')

agg.sort_values(['task', 'avg_quality_score'], ascending=[True, False])